# Session 3: Monte Carlo propagation and its limits
**A. Real-world problem.** The linear formula of Session 2 replaces the formula by a
straight line near the measured values. That is fine for small errors. But what if you
divide by a reading that is close to zero, or sit at the top of a curve where the slope
is zero? We let the computer try 100000 possible experiments and look at the outcome.

**Plan.** (1) Monte Carlo propagation, (2) a derivative by central differences,
(3) non-normal inputs, (4) combined and expanded uncertainty, (5) two failure cases.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: Monte Carlo propagation
Draw each input from its distribution (seeded, so results repeat), push every draw through
the formula and look at the list of outputs. A 95 percent interval is the 2.5th to 97.5th
percentile of that list; it needs no derivative and no normal assumption.

In [ ]:
def mc_samples(f, means, sigmas, n=100000, seed=7):
    """Output samples of f for independent normal inputs."""
    rng = np.random.default_rng(seed)
    xs = [m + s * rng.standard_normal(n) for m, s in zip(means, sigmas)]
    return f(*xs)


def percentile_interval(samples, level=0.95):
    """Central interval holding `level` of the samples."""
    a = 100.0 * (1.0 - level) / 2.0
    lo, hi = np.percentile(samples, [a, 100.0 - a])
    return float(lo), float(hi)

## Step 2: a derivative by central differences
The slope of f at x is (f(x + h) - f(x - h)) / (2 h). A step h that is too big blurs the
curve; one that is too small drowns the difference in round-off. A relative step of about
1e-5 to 1e-6 works well for smooth functions.

In [ ]:
def central_diff(f, x, h):
    return (f(x + h) - f(x - h)) / (2.0 * h)


def numeric_linear_sigma(f, means, sigmas, rel_step=1e-6):
    """First-order sigma using central-difference partial derivatives (independent inputs)."""
    means = [float(m) for m in means]
    var = 0.0
    for i, (m, s) in enumerate(zip(means, sigmas)):
        h = rel_step * (abs(m) if m != 0 else 1.0)

        def fi(v, i=i):
            a = list(means)
            a[i] = v
            return f(*a)

        var += (central_diff(fi, m, h) * s) ** 2
    return float(np.sqrt(var))

## Step 3: inputs that are not normal, and the GUM combination
A digital ruler shows a value rounded to a step LSB (say 0.01 mm). The true value is
anywhere in a band of width LSB: a uniform distribution, with standard deviation
LSB / sqrt(12). Independent standard uncertainties combine in quadrature (after being
multiplied by their sensitivities) into the combined uncertainty u_c; the expanded
uncertainty is U = k u_c, with k = 2 for about 95 percent coverage.

In [ ]:
def uniform_sigma(lsb):
    return lsb / np.sqrt(12.0)


def combined_uncertainty(sens, us):
    return float(np.sqrt(np.sum((np.asarray(sens, dtype=float) * np.asarray(us, dtype=float)) ** 2)))


def expanded(uc, k=2.0):
    return k * uc

## Step 4: when the linear formula fails
(a) A ratio with the denominator near zero. (b) A function at its maximum: the slope is
zero, so the first-order formula says "no uncertainty". The second-order term gives
sigma = |f''| s^2 / sqrt(2) for a normal input of spread s.

In [ ]:
def ratio(x, y):
    return x / y


def second_order_sigma(f2, s):
    """Spread of f near a maximum or minimum for a normal input: |f''| s^2 / sqrt(2)."""
    return abs(f2) * s ** 2 / np.sqrt(2.0)

## Step 5: see it
Figure 1: Monte Carlo histogram of x / y with the linear Gaussian on top.
Figure 2: the central-difference error against the step size.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    for rel in (0.01, 0.30):
        out = mc_samples(ratio, [10.0, 5.0], [0.5, 5.0 * rel])
        lin = numeric_linear_sigma(ratio, [10.0, 5.0], [0.5, 5.0 * rel])
        lo, hi = percentile_interval(out)
        print(f"y error {100 * rel:.0f}%: linear {2 - 1.96 * lin:.3f}..{2 + 1.96 * lin:.3f}, "
              f"Monte Carlo {lo:.3f}..{hi:.3f}")
    out = mc_samples(ratio, [10.0, 5.0], [0.5, 1.5])
    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.hist(out, bins=np.linspace(-1, 8, 181), density=True, alpha=0.5, label="Monte Carlo (100000)")
    lin = numeric_linear_sigma(ratio, [10.0, 5.0], [0.5, 1.5])
    xs = np.linspace(-1, 8, 400)
    ax.plot(xs, np.exp(-0.5 * ((xs - 2) / lin) ** 2) / (lin * np.sqrt(2 * np.pi)), "r", label="linear formula")
    ax.set_xlabel("x / y (no unit)")
    ax.set_ylabel("probability density")
    ax.set_title("30 % error in the denominator: skewed output")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "ratio_skew.png", dpi=120)

    f = lambda t: 4 * np.pi ** 2 * 1.0 / t ** 2
    exact = -8 * np.pi ** 2 / 2.007 ** 3
    hs = 10.0 ** -np.arange(1, 14, dtype=float)
    errs = [abs(central_diff(f, 2.007, h * 2.007) / exact - 1) for h in hs]
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.loglog(hs, errs, "o-")
    ax2.set_xlabel("relative step size h / T (no unit)")
    ax2.set_ylabel("relative error of the derivative (no unit)")
    ax2.set_title("Central difference: truncation versus round-off")
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "step_size.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the exercises, answer the questions in the README (section F), run the checkpoint,
then start the mini-project (error budget for a metal cylinder).

## Coding tasks
1. **Easy.** Change the denominator error in `lesson.py` to 10 percent and print both intervals; compare with the answer to question D in Section F.
2. **Stretch.** Use `central_diff` on sin(x) at x = 1 with steps 1e-1, 1e-3 and 1e-6 and compare with cos(1); find the best step.
3. **Explain.** Explain, in your own words, why a function with a maximum has zero first-order uncertainty but nonzero true uncertainty, and what the second-order term adds.

## Assessment
- **A. Conceptual.** Why does the linear formula fail for a ratio when the denominator has a large relative error, and why is the true interval skewed to the right?
- **B. Numerical.** A digital caliper shows steps of 0.1 mm. Find the standard uncertainty from the resolution and the expanded uncertainty with k = 2.
- **C. Graph and simulation interpretation.** In the first figure, estimate by eye where the upper 95 percent end is for the histogram and for the Gaussian. What does the difference mean for a lab report?
- **D. Design and optimisation challenge.** Design: for the ratio x / y with a 5 percent error in x, at which relative errors of y is the linear upper end within 5 percent of the Monte Carlo upper end? Test 10 percent and 30 percent.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    # small errors: linear and Monte Carlo agree
    o1 = L.mc_samples(L.ratio, [10.0, 5.0], [0.5, 0.05])
    lin1 = L.numeric_linear_sigma(L.ratio, [10.0, 5.0], [0.5, 0.05])
    lo1, hi1 = L.percentile_interval(o1)
    checks.append(("1 % error in the denominator: linear and Monte Carlo intervals both 1.80 to 2.20",
                   abs(2 - 1.96 * lin1 - 1.80) < 5e-3 and abs(lo1 - 1.80) < 5e-3 and abs(hi1 - 2.20) < 5e-3))
    # worked example: large error
    o = L.mc_samples(L.ratio, [10.0, 5.0], [0.5, 1.5])
    lin = L.numeric_linear_sigma(L.ratio, [10.0, 5.0], [0.5, 1.5])
    lo, hi = L.percentile_interval(o)
    checks.append(("worked example: linear sigma 0.61, linear interval 0.81 to 3.19",
                   abs(lin - 0.608) < 5e-3 and abs(2 - 1.96 * lin - 0.81) < 5e-3 and abs(2 + 1.96 * lin - 3.19) < 5e-3))
    hand = 2.0 * np.hypot(0.5 / 10.0, 1.5 / 5.0)
    checks.append(("hand calculation: ratio relative errors 5 % and 30 % give sigma 2 * 0.304 = 0.61", abs(hand - lin) < 1e-4))
    checks.append(("worked example: Monte Carlo 95 % interval is 1.24 to 4.85 (asymmetric)",
                   abs(lo - 1.24) < 0.03 and abs(hi - 4.85) < 0.06 and (hi - 2.0) > 2 * (2.0 - lo)))
    cover = float(np.mean((o > 2 - 1.96 * lin) & (o < 2 + 1.96 * lin)))
    checks.append(("the linear 95 % interval holds only about 89 % of the Monte Carlo outcomes", abs(cover - 0.894) < 0.006))
    checks.append(("the Monte Carlo mean of x/y is above 2 (about 2.18), the linear value", 2.12 < o.mean() < 2.24))
    checks.append(("seeded Monte Carlo is repeatable", np.array_equal(L.mc_samples(L.ratio, [10.0, 5.0], [0.5, 1.5], n=50),
                                                                   L.mc_samples(L.ratio, [10.0, 5.0], [0.5, 1.5], n=50))))
    # central differences
    f = lambda t: 4 * np.pi ** 2 / t ** 2
    exact = -8 * np.pi ** 2 / 2.007 ** 3
    err = lambda h: abs(L.central_diff(f, 2.007, h * 2.007) / exact - 1)
    checks.append(("central difference: analytic slope is -9.77 and a step of 1e-6 is right to 1e-11",
                   abs(exact + 9.767) < 5e-3 and err(1e-6) < 1e-11))
    checks.append(("step size: error 2e-2 at h = 0.1, 2e-6 at 1e-3, and round-off makes 1e-13 worse than 1e-6",
                   abs(err(0.1) - 0.0203) < 5e-4 and abs(err(1e-3) - 2.0e-6) < 1e-7 and err(1e-13) > 1e3 * err(1e-6)))
    # non-normal inputs, GUM
    checks.append(("digital ruler: LSB 0.01 mm gives standard uncertainty 0.0029 mm", abs(L.uniform_sigma(0.01) - 0.00289) < 5e-6))
    u = np.random.default_rng(1).uniform(-0.005, 0.005, 10 ** 6)
    checks.append(("simulation of a uniform band of width LSB agrees with LSB / sqrt(12)", abs(u.std() / L.uniform_sigma(0.01) - 1) < 0.01))
    z = np.random.default_rng(3).standard_normal(10 ** 6)
    checks.append(("coverage factor k = 2 covers 95.45 % of a normal distribution", abs(np.mean(np.abs(z) < 2) - 0.9545) < 0.002 and L.expanded(0.5) == 1.0))
    checks.append(("combined uncertainty of 3 and 4 (sensitivity 1) is 5", abs(L.combined_uncertainty([1, 1], [3, 4]) - 5.0) < 1e-12))
    # failure case: maximum
    s = 0.1
    sm = L.mc_samples(np.sin, [np.pi / 2], [s])
    lin_top = L.numeric_linear_sigma(np.sin, [np.pi / 2], [s])
    checks.append(("failure: at the top of sin the first-order sigma is 0 but the true spread is 0.0070",
                   lin_top < 1e-9 and abs(sm.std() - 0.0070) < 2e-4))
    checks.append(("second-order formula |f''| s^2 / sqrt(2) = 0.0071 matches, and the mean drops to 0.995",
                   abs(L.second_order_sigma(-1.0, s) - 0.0071) < 5e-5 and abs(sm.std() / L.second_order_sigma(-1.0, s) - 1) < 0.02
                   and abs(sm.mean() - 0.995) < 5e-4))
    sm1 = L.mc_samples(np.sin, [1.0], [s])
    checks.append(("away from the top (x = 1 rad) the first-order formula works: 0.054",
                   abs(sm1.std() - np.cos(1.0) * s) < 5e-4 and abs(np.cos(1.0) * s - 0.054) < 5e-4))
    # mini-project: density of a cylinder from calipers and a scale
    dr = np.array([12.71, 12.69, 12.70, 12.72, 12.68])
    hr = np.array([50.02, 49.98, 50.01, 50.03, 49.96])
    m = 17.10
    ud = np.hypot(dr.std(ddof=1) / np.sqrt(5), L.uniform_sigma(0.01))
    uh = np.hypot(hr.std(ddof=1) / np.sqrt(5), L.uniform_sigma(0.01))
    um = np.hypot(L.uniform_sigma(0.01), 0.01)
    rho = lambda mm, d, h: mm / (np.pi * (d / 20.0) ** 2 * (h / 10.0))
    d, h = dr.mean(), hr.mean()
    v = rho(m, d, h)
    uc = L.numeric_linear_sigma(rho, [m, d, h], [um, ud, uh])
    checks.append(("mini-project: standard uncertainties of d, h, m are 0.0076 mm, 0.0134 mm, 0.0104 g",
                   abs(ud - 0.00764) < 5e-5 and abs(uh - 0.01335) < 5e-5 and abs(um - 0.01041) < 5e-5))
    checks.append(("mini-project: density 2.6998 g/cm^3, combined uncertainty 0.0037, expanded (k = 2) 0.0074",
                   abs(v - 2.6998) < 5e-4 and abs(uc - 0.00371) < 5e-5 and abs(L.expanded(uc) - 0.00742) < 1e-4))
    shares = np.array([um / m, 2 * ud / d, uh / h]) ** 2
    shares = shares / shares.sum()
    checks.append(("mini-project: variance shares mass 20 %, diameter 77 %, height 4 %; diameter dominates",
                   np.allclose(shares, [0.196, 0.766, 0.038], atol=0.002) and int(np.argmax(shares)) == 1))
    mc = L.mc_samples(rho, [m, d, h], [um, ud, uh])
    mlo, mhi = L.percentile_interval(mc)
    checks.append(("mini-project: Monte Carlo agrees: sigma 0.0037, 95 % interval 2.693 to 2.707",
                   abs(mc.std() / uc - 1) < 0.02 and abs(mlo - 2.6925) < 5e-4 and abs(mhi - 2.7070) < 5e-4))
    checks.append(("mini-project: the final report is 2.700 +- 0.007 g/cm^3 (k = 2)",
                   (round(v, 3), round(2 * uc, 3)) == (2.700, 0.007)))
    # assessment numbers
    checks.append(("question B: 0.1 mm step gives 0.0289 mm and U = 0.0577 mm",
                   abs(L.uniform_sigma(0.1) - 0.0289) < 5e-5 and abs(L.expanded(L.uniform_sigma(0.1)) - 0.0577) < 5e-5))
    o10b = L.mc_samples(L.ratio, [10.0, 5.0], [0.5, 0.5])
    l10 = L.numeric_linear_sigma(L.ratio, [10.0, 5.0], [0.5, 0.5])
    m10 = L.percentile_interval(o10b)
    d10 = (m10[1] - (2 + 1.96 * l10)) / m10[1]
    d30 = (hi - (2 + 1.96 * lin)) / hi
    checks.append(("question D: at 10 % linear is 1.56 to 2.44, Monte Carlo 1.62 to 2.53, upper ends differ 3.6 %; at 30 % they differ 34 %",
                   abs(2 - 1.96 * l10 - 1.56) < 5e-3 and abs(2 + 1.96 * l10 - 2.44) < 5e-3 and abs(m10[0] - 1.62) < 5e-3
                   and abs(m10[1] - 2.53) < 5e-3 and abs(d10 - 0.036) < 0.003 and abs(d30 - 0.34) < 0.01))
    checks.append(("app challenge: (1.96 r)^2 < 0.05 holds at 10 % (0.038) and fails at 12 % (0.055) and 25 %",
                   (1.96 * 0.10) ** 2 < 0.05 < (1.96 * 0.12) ** 2 and abs((1.96 * 0.10) ** 2 - 0.038) < 5e-4))
    checks.append(("misconception: central-difference error is 4e-12 at 1e-6 and 2e-5 at 1e-13",
                   abs(np.log10(err(1e-6)) - np.log10(4.3e-12)) < 0.2 and abs(np.log10(err(1e-13)) - np.log10(1.7e-5)) < 0.3))
    # milestone data quoted in the roadmap
    gf = lambda a, b: 4 * np.pi ** 2 * a / b ** 2
    ref = L.mc_samples(gf, [1.0, 2.007], [0.005, 0.010], n=10 ** 6).std()
    e200 = np.mean([abs(L.mc_samples(gf, [1.0, 2.007], [0.005, 0.010], n=200, seed=k).std() / ref - 1) for k in range(200)])
    checks.append(("roadmap fact: plain Monte Carlo with N = 200 misses the spread of g by about 4 percent",
                   abs(ref - 0.109) < 1e-3 and 0.035 < e200 < 0.047))
    o10 = L.mc_samples(L.ratio, [10.0, 5.0], [0.5, 0.5])
    checks.append(("roadmap fact: for a 10 % denominator error the mean of x/y is 2.02 while the linear value is 2",
                   abs(o10.mean() - 2.02) < 0.004))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
